## Data Formatting & Expansion

In [ ]:
import pandas as pd

# Load the dataset
file_path = "Cleaned_Word_sense.xlsx"
df = pd.read_excel(file_path, sheet_name="Cleaned Word Sense")

In [ ]:
# Create a new DataFrame with the required structure
formatted_df = pd.DataFrame({
    "FOS/Expression": df["Fos"],
    "Meaning": df["Meaning_en"],
    "Non-Literal Usage": df["Examples"],
    "Literal Usage": None,
    "Similarity Score": None
})

# Save the formatted dataset
# formatted_file_path = "Formatted_Dataset.xlsx"
# formatted_df.to_excel(formatted_file_path, index=False)

# print(f"Formatted dataset saved as {formatted_file_path}")

In [ ]:
formatted_df.head()

,FOS/Expression,Meaning,Non-Literal Usage,Literal Usage,Similarity Score
0,alsa og balay,said to self or someone when the meal includes...,Siya'y alsa og balay kada buntag aron makataba...,None,None
1,ang baka nisulod sa kulon,used to tell someone that he is a liar or you ...,"Sa dihang nagluto siya, naay baka nisulod sa k...",None,None
2,ang masuya madeads,used to tell someone not to criticize or hate ...,"Si Maria kay permi nalang masuya, maong ang ma...",None,None
3,ayaw pagbuot kay lain-lain ta og lubot,don't tell me what to d,"Ayaw pagbuot kay lain-lain ta og lubot, respet...",None,None
4,"ayaw pagbuot, lain-lain tag lubot",don't tell me what to d,"Ayaw pagbuot, lain-lain tag lubot, tanan kita ...",None,None


In [ ]:
# Initialize an empty list to store the processed rows
expanded_rows = []

# Iterate through each row in the dataframe
for _, row in formatted_df.iterrows():
    fos = row["FOS/Expression"]
    meaning = row["Meaning"]
    examples = str(row["Non-Literal Usage"]).split(";")  # Split non-literal examples by semicolon

    for example in examples:
        example = example.strip()  # Remove extra spaces
        if example:  # Ensure it's not an empty string
            expanded_rows.append([fos, meaning, example, None, None])

# Create a new dataframe with the expanded rows
expanded_df = pd.DataFrame(expanded_rows, columns=["FOS/Expression", "Meaning","Non-Literal Usage", "Literal Usage", "Similarity Score"])

# Save the new dataset
expanded_file_path = "Expanded_Dataset.xlsx"
expanded_df.to_excel(expanded_file_path, index=False)

print(f"Expanded dataset saved as {expanded_file_path}")


Expanded dataset saved as Expanded_Dataset.xlsx


In [ ]:
expanded_df.head()

,FOS/Expression,Meaning,Non-Literal Usage,Literal Usage,Similarity Score
0,alsa og balay,said to self or someone when the meal includes...,Siya'y alsa og balay kada buntag aron makataba...,None,None
1,alsa og balay,said to self or someone when the meal includes...,Ang mga bata sa bukid kay alsa og balay para m...,None,None
2,alsa og balay,said to self or someone when the meal includes...,"Sa dihang naglisod sila, alsa og balay ang tan...",None,None
3,alsa og balay,said to self or someone when the meal includes...,"Bisan kapoy na, alsa og balay gihapon siya par...",None,None
4,alsa og balay,said to self or someone when the meal includes...,Ang mga magtiayon nag-alsa og balay para sa il...,None,None


In [ ]:
!pip install pandas openai openpyxl tqdm python-dotenv

In [ ]:
# importing os module for environment variables
import os
# importing necessary functions from dotenv library
from dotenv import load_dotenv, dotenv_values
# loading variables from .env file
load_dotenv()

In [ ]:
openai.api_key = os.getenv("OPENAI_API_KEY")

In [ ]:
# from openai import OpenAI
# client = OpenAI()
# completion = client.chat.completions.create(
#    model="gpt-4o",
#    store=True,
#    messages=[
#        {"role": "user", "content": "write a haiku about ai"}
#    ]
#)


## Data Augmentation

In [ ]:
# Load the expanded dataset
file_path = "Expanded_Dataset.xlsx"
df = pd.read_excel(file_path)

In [ ]:
import openai
from tqdm import tqdm

# Function to generate literal usage using GPT-4o
def generate_literal_usage(fos_expression, meaning, non_literal_example):
    prompt = f"""
    Given the Cebuano expression: "{fos_expression}" with the figurative meaning: "{meaning}",
    and a non-literal usage example: "{non_literal_example}",
    generate a sentence where the expression is used in its **literal, non-figurative** sense.
    The sentence should be simple and natural.
    """

    response = openai.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": "You are an AI that generates literal usage examples for given Cebuano expressions."},
            {"role": "user", "content": prompt}
        ]
    )

    return response.choices[0].message.content.strip()

In [ ]:
# Generate literal examples for each row
literal_examples = []
for _, row in tqdm(df.iterrows(), total=len(df), desc="Generating Literal Examples"):
    fos = row["FOS/Expression"]
    meaning = row["Meaning"]
    non_literal_example = row["Non-Literal Usage"]

    try:
        literal_example = generate_literal_usage(fos, meaning, non_literal_example)
    except Exception as e:
        print(f"Error generating for {fos}: {e}")
        literal_example = None  # Handle errors gracefully

    literal_examples.append(literal_example)

# Add generated literal examples to the dataframe
df["Literal Usage"] = literal_examples
df.head()

Generating Literal Examples: 100%|██████████| 805/805 [14:55<00:00,  1.11s/it]


,FOS/Expression,Meaning,Non-Literal Usage,Literal Usage,Similarity Score
0,alsa og balay,said to self or someone when the meal includes...,Siya'y alsa og balay kada buntag aron makataba...,Nag-alsa sila og balay ngadto sa pikas baranga...,NaN
1,alsa og balay,said to self or someone when the meal includes...,Ang mga bata sa bukid kay alsa og balay para m...,Gihangyo sa pamilya ang ilang mga silingan nga...,NaN
2,alsa og balay,said to self or someone when the meal includes...,"Sa dihang naglisod sila, alsa og balay ang tan...",Nag-alsa og balay ang pamilya kay mobalhin sil...,NaN
3,alsa og balay,said to self or someone when the meal includes...,"Bisan kapoy na, alsa og balay gihapon siya par...",Nanginahanglan sila og dugang nga tawo kay kin...,NaN
4,alsa og balay,said to self or someone when the meal includes...,Ang mga magtiayon nag-alsa og balay para sa il...,Nagkauban mi sa akong mga amigo aron alsa og b...,NaN


In [ ]:
# Save the new dataset
file_path = "../../Dataset/Full_Dataset.xlsx"
df.to_excel(file_path, index=False)

print(f"Dataset saved as {file_path}")

Dataset saved as Full_Dataset.xlsx
